# 課題5：映画レビューの評判分析

本課題ではAmazon傘下の「IMDb」に投稿された映画のレビュー（英語）を分析し、レビューがPositive（ポジティブ）か、Negative（ネガティブ）かの判別を行ないます。

データセットは、以下のサイトで配布されているものを利用します。

[Large Movie Review Dataset](https://ai.stanford.edu/%7Eamaas/data/sentiment/)

わからない場合は、ここまでのレッスン内容や各種ライブラリの公式ドキュメントを参照しましょう。

## 1. 必要なライブラリのimport

In [25]:
# （変更しないでください）

# 必要なライブラリのimport
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 文章ファイル検索用
import glob
import collections
from sklearn.feature_extraction import DictVectorizer

# DataFrameですべての列を表示する設定
pd.options.display.max_columns = None

# seabornによる装飾を適用する
sns.set_theme()

## 2. データの読み込み

In [ ]:
# ダウンロードした圧縮ファイルを解凍する（変更しないでください）
!tar zxvf aclImdb_v1.tar.gz

*./aclImdb* フォルダ内にあるファイルを読み込みます。

In [26]:
# trainフォルダのファイル一覧を取得（変更しないでください）
train_neg_files = glob.glob("./aclImdb/train/neg/*")
train_pos_files = glob.glob("./aclImdb/train/pos/*")

# testフォルダのファイル一覧を取得（変更しないでください）
test_neg_files = glob.glob("./aclImdb/test/neg/*")
test_pos_files = glob.glob("./aclImdb/test/pos/*")

In [ ]:
# それぞれのファイル数を確認
print(
    len(train_neg_files),
    len(train_pos_files),
    len(test_neg_files),
    len(test_pos_files)
)

前処理をするため、合計50000あるファイルをリストにまとめます。

In [9]:
# ファイル名をまとめたリストを用意（変更しないでください）
filenames = train_neg_files + train_pos_files + test_neg_files + test_pos_files

# filenamesの長さを確認（変更しないでください）
len(filenames)

50000

リストの最初と最後のファイルを確認してみます。

In [5]:
# エンコーディング用定数（変更しないでください）
ENCODING = 'utf-8'

In [ ]:
# 最初のファイルの内容を確認
with open(filenames[0], "r", encoding=ENCODING) as f:
    first = f.read()
    print(first)

In [ ]:
# 最後のファイルの内容を確認
with open(filenames[-1], "r", encoding=ENCODING) as f:
    last = f.read()
    print(last)

## 3. データの前処理

データの前処理として、形態素解析と行列への変換を行ないます。

### 形態素解析

In [28]:
# 文字列の中で使われている単語ごとの数を返す関数を作成
#（レッスン本編の内容を確認して、下記にコードを追記してください）
def get_word_count(text, min_length=3):
    # ノイズの除去：不要と思われる文字を除去する
    for ch in ".,:;!?-+*/=()[]{}<>~^#$@%&'\"_0123456789":
        text = text.replace(ch, ' ')

    # 形態素解析：文章を単語に分割
    _words = text.strip().split()

    # 表記のゆれの補正：
    # 単語のリストを受け取り、指定された文字数以上の単語だけをすべて小文字にして返す
    _words = [_word.lower() for _word in _words if len(_word) >= min_length]

    # collections.Counterの戻り値は辞書型のサブクラス
    _count = collections.Counter(_words)

    # 辞書型に変換して返す
    return dict(_count)

In [ ]:
# 最初のファイルを使って、先ほど作成した関数をテスト
with open(filenames[0], 'r', encoding=ENCODING) as f:
    text = f.read()

get_word_count(text)

In [30]:
# 単語ごとの数のリストを作成（変更しないでください）
word_count_data = []

In [31]:
# すべてのファイルに対して、先ほど作成した関数を実行
for filename in filenames:
    with open(filename, 'r', encoding=ENCODING) as f:
        text = f.read()
        count = get_word_count(text)
        word_count_data.append(count)

In [ ]:
# 単語ごとの数のリストの長さを確認
len(word_count_data)

In [ ]:
# 単語ごとの数のリストの0番目を表示
word_count_data[0]

### 行列への変換

In [32]:
# DictVectorizerを使用して行列に変換し、datasetに格納する
vec = DictVectorizer()
dataset = vec.fit_transform(word_count_data)

In [ ]:
# datasetの大きさを確認
dataset.shape

In [ ]:
# 各列に対応した単語を取得
vec.get_feature_names_out()

## 4. 機械学習の実施

In [33]:
# 必要なライブラリの追加import（変更しないでください）
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

目的変数と説明変数を用意します。

In [34]:
# 目的変数Yの用意
# neg12500 + pos12500 + neg12500 + pos12500 = 50000
Y = [0]*12500 + [0]*12500 + [1]*12500 + [1]*12500

In [35]:
# 上記のY、および前処理されたdatasetからデータを分割し、
# X_train, Y_train, X_test, Y_testに格納する
#
# 詳細：
#   - dataset の先頭から25000件を 変数 X_train に、残りを変数 X_test に代入
#   - 目的変数 Y の先頭から25000件を 変数 Y_train に、残りを Y_test に代入
X_train = dataset[:25000]
X_test = dataset[25000:]
Y_train = Y[:25000]
Y_test = Y[25000:]

In [36]:
# X_trainとY_trainを、train_test_splitで7:3に分割し、3割のほうを検証データ（X_valid, Y_valid）にする
X_train, X_valid, Y_train, Y_valid = train_test_split(X_train, Y_train, test_size=0.3, random_state=0)

In [ ]:
# ロジスティック回帰モデルを作成し、学習して、検証データによる予測を実施する
logistic_model = LogisticRegression(max_iter=2000)
logistic_model.fit(X_train, Y_train)
Y_pred = logistic_model.predict(X_valid)

# classification_reportを実行し、検証データによるモデルの評価を行なう
print(classification_report(Y_valid, Y_pred))

## 5. テストデータによる評価

最後に、テストデータで評価を行ないましょう。

In [ ]:
# テストデータで予測を実施する


# classification_reportを実行し、テストデータによるモデルの評価を行なう
